# Pré-processamento do Dataset de Soja

Nesta etapa vamos preparar os dados para o modelo de visão computacional.

O fluxo será:

1. Baixar os datasets via KaggleHub;
2. Inspecionar as pastas originais;
3. Organizar as imagens em uma estrutura única por classe;
4. Criar divisão treino, validação e teste;
5. Reduzir excesso da classe saudável no treino;
6. Aplicar augmentação offline nas classes doentes;
7. Conferir a distribuição final;
8. Preparação dos dados para treinamento.

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
from src.preprocessing.soja_preprocessing import (
    baixar_datasets,
    listar_pastas_dataset,
    contar_imagens_por_classe,
    organizar_dataset_soja,
    mostrar_distribuicao_dataset,
    criar_split_dataset,
    limitar_classe_saudavel_treino,
    aplicar_augmentacao_treino,
    mostrar_distribuicao_split,
    DATASET_ORGANIZADO,
    DATASET_SPLIT
)

from src.dataset.soja_dataset import (
    criar_datasets,
    criar_dataloaders,
    mostrar_info_datasets,
    obter_classes,
    obter_class_to_idx,
    obter_idx_to_class,
    DATASET_SPLIT
)

## 1. Download dos datasets

Vamos baixar dois datasets:

- Dataset de folhas de soja doentes;
- Dataset PlantVillage, usado para obter imagens de soja saudável.

In [0]:
pasta_soja_doente, pasta_plantvillage = baixar_datasets()

Baixando dataset de soja doente...


100%|██████████| 1.93G/1.93G [00:24<00:00, 83.8MB/s]

Extracting files...


Baixando dataset PlantVillage...


100%|██████████| 818M/818M [00:06<00:00, 133MB/s]

Extracting files...



Downloads finalizados:
Soja doente: /home/spark-2fec6790-c057-491e-b157-fb/.cache/kagglehub/datasets/sivm205/soybean-diseased-leaf-dataset/versions/1
PlantVillage: /home/spark-2fec6790-c057-491e-b157-fb/.cache/kagglehub/datasets/mohitsingh1804/plantvillage/versions/1


## 2. Inspeção inicial dos datasets

Antes de transformar os dados, é importante verificar a estrutura das pastas originais.

Isso ajuda a confirmar se os caminhos e nomes das classes estão corretos.

In [0]:
listar_pastas_dataset(pasta_soja_doente)


Pastas encontradas em /home/spark-2fec6790-c057-491e-b157-fb/.cache/kagglehub/datasets/sivm205/soybean-diseased-leaf-dataset/versions/1:
Mossaic Virus
Southern blight
Sudden Death Syndrone
Yellow Mosaic
bacterial_blight
brown_spot
crestamento
ferrugen
powdery_mildew
septoria


In [0]:
listar_pastas_dataset(pasta_plantvillage)


Pastas encontradas em /home/spark-2fec6790-c057-491e-b157-fb/.cache/kagglehub/datasets/mohitsingh1804/plantvillage/versions/1:
PlantVillage


In [0]:
listar_pastas_dataset(pasta_plantvillage / "PlantVillage")


Pastas encontradas em /home/spark-2fec6790-c057-491e-b157-fb/.cache/kagglehub/datasets/mohitsingh1804/plantvillage/versions/1/PlantVillage:
train
val


In [0]:
listar_pastas_dataset(pasta_plantvillage / "PlantVillage" / "train")


Pastas encontradas em /home/spark-2fec6790-c057-491e-b157-fb/.cache/kagglehub/datasets/mohitsingh1804/plantvillage/versions/1/PlantVillage/train:
Apple___Apple_scab
Apple___Black_rot
Apple___Cedar_apple_rust
Apple___healthy
Blueberry___healthy
Cherry_(including_sour)___Powdery_mildew
Cherry_(including_sour)___healthy
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
Corn_(maize)___Common_rust_
Corn_(maize)___Northern_Leaf_Blight
Corn_(maize)___healthy
Grape___Black_rot
Grape___Esca_(Black_Measles)
Grape___Leaf_blight_(Isariopsis_Leaf_Spot)
Grape___healthy
Orange___Haunglongbing_(Citrus_greening)
Peach___Bacterial_spot
Peach___healthy
Pepper,_bell___Bacterial_spot
Pepper,_bell___healthy
Potato___Early_blight
Potato___Late_blight
Potato___healthy
Raspberry___healthy
Soybean___healthy
Squash___Powdery_mildew
Strawberry___Leaf_scorch
Strawberry___healthy
Tomato___Bacterial_spot
Tomato___Early_blight
Tomato___Late_blight
Tomato___Leaf_Mold
Tomato___Septoria_leaf_spot
Tomato___Spider_mites

## 3. Organização do dataset

Agora vamos criar um dataset único de soja.

As imagens serão organizadas assim:

```
dataset_soja/
    Soja_Saudavel/
    Ferrugem/
    Mancha_Parda/
    Oidio/
    ...
```

Nesta etapa também fazemos a padronização dos nomes das classes.

In [0]:
organizar_dataset_soja(
    pasta_soja_doente=pasta_soja_doente,
    pasta_plantvillage=pasta_plantvillage,
    limpar_destino=True
)


Organizando dataset de soja...
Soja saudável copiada: 5090 imagens
ferrugen -> Ferrugem: 65 imagens
Yellow Mosaic -> Mosaico_Amarelo: 110 imagens
brown_spot -> Mancha_Parda: 81 imagens
powdery_mildew -> Oidio: 137 imagens
septoria -> Septoriose: 21 imagens
Southern blight -> Podridao_Sul: 62 imagens
Mossaic Virus -> Virus_Mosaico: 22 imagens
Sudden Death Syndrone -> Sindrome_Morte_Subita: 110 imagens
bacterial_blight -> Queima_Bacteriana: 88 imagens
crestamento -> Crestamento: 5 imagens

Dataset organizado criado em:
/Volumes/workspace/agro-diasease/soja/dataset_soja


In [0]:
mostrar_distribuicao_dataset(DATASET_ORGANIZADO)


Distribuição de imagens em: /Volumes/workspace/agro-diasease/soja/dataset_soja
Crestamento: 5
Ferrugem: 65
Mancha_Parda: 81
Mosaico_Amarelo: 110
Oidio: 137
Podridao_Sul: 62
Queima_Bacteriana: 88
Septoriose: 21
Sindrome_Morte_Subita: 110
Soja_Saudavel: 8256
Virus_Mosaico: 22


## 4. Split treino, validação e teste

Agora vamos dividir o dataset em:

- `train`: usado para treinar o modelo;
- `val`: usado para acompanhar o desempenho durante o treinamento;
- `test`: usado somente para avaliação final.

A divisão configurada no arquivo é:

- 70% treino;
- 15% validação;
- 15% teste.

In [0]:
criar_split_dataset(
    pasta_origem=DATASET_ORGANIZADO,
    pasta_destino=DATASET_SPLIT,
    limpar_destino=True
)


Criando split train/val/test...
Soja_Saudavel: train=3563, val=763, test=764
Ferrugem: train=45, val=9, test=11
Mosaico_Amarelo: train=77, val=16, test=17
Mancha_Parda: train=56, val=12, test=13
Oidio: train=95, val=20, test=22
Septoriose: train=14, val=3, test=4
Podridao_Sul: train=43, val=9, test=10
Virus_Mosaico: train=15, val=3, test=4
Sindrome_Morte_Subita: train=77, val=16, test=17
Queima_Bacteriana: train=61, val=13, test=14
Crestamento: train=3, val=1, test=1

Dataset com split criado em:
/Volumes/workspace/agro-diasease/soja/dataset_soja_split


In [0]:
mostrar_distribuicao_split(DATASET_SPLIT)


Distribuição final por split:

TRAIN
Crestamento: 3
Ferrugem: 45
Mancha_Parda: 56
Mosaico_Amarelo: 77
Oidio: 95
Podridao_Sul: 43
Queima_Bacteriana: 61
Septoriose: 14
Sindrome_Morte_Subita: 77
Soja_Saudavel: 5513
Virus_Mosaico: 15

VAL
Crestamento: 1
Ferrugem: 9
Mancha_Parda: 12
Mosaico_Amarelo: 16
Oidio: 20
Podridao_Sul: 9
Queima_Bacteriana: 13
Septoriose: 3
Sindrome_Morte_Subita: 16
Soja_Saudavel: 763
Virus_Mosaico: 3

TEST
Crestamento: 1
Ferrugem: 11
Mancha_Parda: 13
Mosaico_Amarelo: 17
Oidio: 22
Podridao_Sul: 10
Queima_Bacteriana: 14
Septoriose: 4
Sindrome_Morte_Subita: 17
Soja_Saudavel: 764
Virus_Mosaico: 4


## 5. Redução da classe saudável no treino

A classe saudável pode ter muito mais imagens do que algumas classes doentes.

Para reduzir o desbalanceamento inicial, vamos limitar a quantidade de imagens saudáveis no treino.

In [0]:
limitar_classe_saudavel_treino(
    pasta_split=DATASET_SPLIT,
    max_imagens=100
)


Classe saudável reduzida no treino: 3563 -> 100


## 6. Augmentação offline nas classes doentes

Agora vamos aumentar artificialmente as classes doentes no conjunto de treino.

A augmentação aplica:

- rotação aleatória;
- espelhamento horizontal;
- alteração de brilho.

A classe saudável será ignorada nessa etapa.

In [0]:
aplicar_augmentacao_treino(
    pasta_split=DATASET_SPLIT,
    alvo_por_classe=400
)


Aplicando augmentação offline nas classes doentes...
Ferrugem: 45 -> 400
Mosaico_Amarelo: 77 -> 400
Mancha_Parda: 56 -> 400
Oidio: 95 -> 400
Septoriose: 14 -> 400
Podridao_Sul: 43 -> 400
Virus_Mosaico: 15 -> 400
Sindrome_Morte_Subita: 77 -> 400
Queima_Bacteriana: 61 -> 400
Crestamento: 3 -> 400


## 7. Distribuição final

Por fim, vamos conferir a quantidade final de imagens por classe em cada split.

In [0]:
mostrar_distribuicao_split(DATASET_SPLIT)


Distribuição final por split:

TRAIN
Crestamento: 400
Ferrugem: 400
Mancha_Parda: 400
Mosaico_Amarelo: 400
Oidio: 400
Podridao_Sul: 400
Queima_Bacteriana: 400
Septoriose: 400
Sindrome_Morte_Subita: 400
Soja_Saudavel: 100
Virus_Mosaico: 400

VAL
Crestamento: 1
Ferrugem: 9
Mancha_Parda: 12
Mosaico_Amarelo: 16
Oidio: 20
Podridao_Sul: 9
Queima_Bacteriana: 13
Septoriose: 3
Sindrome_Morte_Subita: 16
Soja_Saudavel: 763
Virus_Mosaico: 3

TEST
Crestamento: 1
Ferrugem: 11
Mancha_Parda: 13
Mosaico_Amarelo: 17
Oidio: 22
Podridao_Sul: 10
Queima_Bacteriana: 14
Septoriose: 4
Sindrome_Morte_Subita: 17
Soja_Saudavel: 764
Virus_Mosaico: 4


## 8. Preparação dos dados para treinamento

Nesta etapa vamos criar os `transforms`, os `ImageFolder` e os `DataLoaders`.

O objetivo é validar se a estrutura final do dataset está correta e se o PyTorch consegue carregar as imagens normalmente.



Agora vamos criar os datasets de treino, validação e teste usando `ImageFolder`.

O `ImageFolder` lê automaticamente a estrutura:

<t></t>

```
dataset_soja_split/
    train/
        classe_1/
        classe_2/
    val/
        classe_1/
        classe_2/
    test/
        classe_1/
        classe_2/

```

In [0]:
dataset_treino, dataset_val, dataset_teste = criar_datasets(
    pasta_split=DATASET_SPLIT,
    img_size=224
)


Vamos verificar:

- quantidade de imagens;
- quantidade de classes;
- nomes das classes;
- ordem das classes usada pelo PyTorch.

In [0]:
mostrar_info_datasets(
    dataset_treino,
    dataset_val,
    dataset_teste
)


Dataset de treino
Total de imagens: 4100
Total de classes: 11
Classes:
0: Crestamento
1: Ferrugem
2: Mancha_Parda
3: Mosaico_Amarelo
4: Oidio
5: Podridao_Sul
6: Queima_Bacteriana
7: Septoriose
8: Sindrome_Morte_Subita
9: Soja_Saudavel
10: Virus_Mosaico

Dataset de validação
Total de imagens: 865
Total de classes: 11
Classes:
0: Crestamento
1: Ferrugem
2: Mancha_Parda
3: Mosaico_Amarelo
4: Oidio
5: Podridao_Sul
6: Queima_Bacteriana
7: Septoriose
8: Sindrome_Morte_Subita
9: Soja_Saudavel
10: Virus_Mosaico

Dataset de teste
Total de imagens: 877
Total de classes: 11
Classes:
0: Crestamento
1: Ferrugem
2: Mancha_Parda
3: Mosaico_Amarelo
4: Oidio
5: Podridao_Sul
6: Queima_Bacteriana
7: Septoriose
8: Sindrome_Morte_Subita
9: Soja_Saudavel
10: Virus_Mosaico


In [0]:
classes = obter_classes(dataset_treino)
class_to_idx = obter_class_to_idx(dataset_treino)

print("Classes:")
print(classes)

print("\nClass to idx:")
print(class_to_idx)

Classes:
['Crestamento', 'Ferrugem', 'Mancha_Parda', 'Mosaico_Amarelo', 'Oidio', 'Podridao_Sul', 'Queima_Bacteriana', 'Septoriose', 'Sindrome_Morte_Subita', 'Soja_Saudavel', 'Virus_Mosaico']

Class to idx:
{'Crestamento': 0, 'Ferrugem': 1, 'Mancha_Parda': 2, 'Mosaico_Amarelo': 3, 'Oidio': 4, 'Podridao_Sul': 5, 'Queima_Bacteriana': 6, 'Septoriose': 7, 'Sindrome_Morte_Subita': 8, 'Soja_Saudavel': 9, 'Virus_Mosaico': 10}


Agora vamos criar os DataLoaders.

In [0]:
loader_treino, loader_val, loader_teste = criar_dataloaders(
    pasta_split=DATASET_SPLIT,
    img_size=224,
    batch_size=32,
    num_workers=0
)

Antes de seguir para o treinamento, vamos carregar um batch para confirmar se:

- as imagens estão sendo lidas corretamente;
- os tensores têm o formato esperado;
- os rótulos estão vindo corretamente.

In [0]:
imagens, labels = next(iter(loader_treino))

print("Formato das imagens:", imagens.shape)
print("Formato dos labels:", labels.shape)
print("Labels do batch:", labels)

Formato das imagens: torch.Size([32, 3, 224, 224])
Formato dos labels: torch.Size([32])
Labels do batch: tensor([2, 2, 6, 9, 0, 8, 5, 8, 0, 3, 3, 6, 2, 2, 2, 0, 5, 0, 3, 4, 4, 2, 6, 2,
        1, 3, 1, 4, 4, 9, 9, 0])


In [0]:
print("Dataset organizado:", DATASET_ORGANIZADO)
print("Dataset com split:", DATASET_SPLIT)

Dataset organizado: /Volumes/workspace/agro-diasease/soja/dataset_soja
Dataset com split: /Volumes/workspace/agro-diasease/soja/dataset_soja_split


# Pré-processamento concluído

Nesta etapa finalizamos:

- organização das imagens;
- padronização das classes;
- split treino/validação/teste;
- balanceamento básico;
- augmentação offline;
- criação dos datasets;
- criação e teste dos DataLoaders.